# Assignment 5 - Dynamic Crawling & Scraping with Selenium

This notebook demonstrates **dynamic web scraping** using **Selenium**, which simulates a real user browsing experience. This is essential for scraping modern websites that load content dynamically with JavaScript.

## Task Overview

1. Use **Selenium** to simulate user browsing behavior
2. Wait **5-10 seconds** between actions (to be respectful to servers)
3. Implement **pagination** - browse through multiple result pages
4. Scrape structured data and display in a **DataFrame**

## Selected Option: IMDB Movie Search

We'll scrape adventure movies from IMDB, extracting:
- **Movie name**
- **Movie length (runtime)**
- **Age restriction (certificate)**
- **User rating**

---
## Setup and Imports

In [ ]:
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd
from IPython.display import display

# Configuration
WAIT_TIME_MIN = 5  # Minimum wait time in seconds
WAIT_TIME_MAX = 10  # Maximum wait time in seconds
MAX_PAGES = 3  # Number of pages to scrape (original + 2 more)

def random_wait(min_sec=WAIT_TIME_MIN, max_sec=WAIT_TIME_MAX):
    """
    Wait a random amount of time between min_sec and max_sec.
    This simulates human browsing behavior and is respectful to servers.
    """
    wait_time = random.uniform(min_sec, max_sec)
    print(f"  Waiting {wait_time:.1f} seconds...")
    time.sleep(wait_time)

---
## Step 1: Initialize Selenium WebDriver

In [ ]:
def setup_driver():
    """
    Configure and return a Chrome WebDriver instance.
    """
    chrome_options = Options()
    chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("--log-level=3")
    chrome_options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": """
            Object.defineProperty(navigator, 'webdriver', {
                get: () => undefined
            })
        """
    })
    
    return driver

print("Setting up Selenium WebDriver...")
driver = setup_driver()
print("WebDriver initialized successfully!")

Setting up Selenium WebDriver...


WebDriver initialized successfully!


---
## Step 2: Define the Scraper Function

This function extracts movie data using multiple selector strategies for robustness.

In [ ]:
import re

def scrape_movie_data(driver):
    """
    Scrape movie data from the current IMDB search results page.
    Uses multiple selector strategies and JavaScript for robustness.
    """
    movies = []

    # Find movie containers
    movie_containers = driver.find_elements(By.CSS_SELECTOR, "li.ipc-metadata-list-summary-item")

    if len(movie_containers) == 0:
        movie_containers = driver.find_elements(By.CSS_SELECTOR, "div.ipc-metadata-list-summary-item")

    if len(movie_containers) == 0:
        print("  Warning: No movie containers found.")
        return movies

    print(f"  Found {len(movie_containers)} movies in the DOM.")

    for i, container in enumerate(movie_containers):
        movie_data = {
            'name': 'N/A',
            'url': 'N/A',
            'runtime': 'N/A',
            'certificate': 'N/A',
            'rating': 'N/A'
        }

        # Extract movie name
        try:
            title_links = container.find_elements(By.CSS_SELECTOR, "a[href*='/title/tt']")
            for link in title_links:
                text = link.text.strip()
                if text and len(text) > 1:
                    movie_data['name'] = text
                    movie_data['url'] = link.get_attribute('href')
                    break
        except:
            pass

        # Extract rating
        try:
            rating_script = """
            var container = arguments[0];
            var rating = container.querySelector('[aria-label^="IMDb rating:"]');
            if (!rating) rating = container.querySelector('.ipc-rating-star');
            if (rating) return rating.textContent.trim().split(/\s+/)[0];
            return null;
            """
            rating = driver.execute_script(rating_script, container)
            if rating and re.match(r'^\d+(\.\d+)?$', str(rating)):
                movie_data['rating'] = str(rating)
        except Exception as e:
            pass

        # Extract runtime & certificate
        try:
            metadata_items = container.find_elements(By.CSS_SELECTOR, ".dli-title-metadata li.ipc-inline-list__item")
            for item in metadata_items:
                text = item.text.strip()
                if re.match(r'^(\d+h(\s+\d+m)?|\d+m)$', text, re.IGNORECASE):
                    movie_data['runtime'] = text
                elif re.match(r'^\d{4}$', text):
                    pass # It's the year
                else:
                    # Anything else in that block is usually the Certificate (e.g., PG-13, R)
                    if movie_data['certificate'] == 'N/A' and text:
                        movie_data['certificate'] = text
        except:
            pass

        movies.append(movie_data)

    return movies

---
## Step 3: Main Scraping Logic with Pagination

In [ ]:
# IMDB adventure movies search URL
BASE_URL = "https://www.imdb.com/search/title/?genres=adventure&explore=genres&title_type=feature&num_votes=25000,&sort=user_rating,desc"

print("=" * 70)
print("STEP 3: Dynamic Crawling with Pagination")
print("=" * 70)

all_movies = []

try:
    print(f"Navigating to BASE_URL: {BASE_URL}")
    driver.get(BASE_URL)
    
    for page in range(1, MAX_PAGES + 1):
        print(f"\n{'='*50}")
        print(f"Scraping Page {page} of {MAX_PAGES}")
        print("=" * 50)
        
        # Wait 5 to 10 seconds before action
        random_wait()
        
        # Scroll to load dynamic content
        print("  Scrolling to load images...")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        random_wait(2, 3)
        
        # Scrape movies currently in DOM
        current_movies = scrape_movie_data(driver)
        
        # We only want the newly added movies for this page
        new_movies = current_movies[len(all_movies):]
        all_movies.extend(new_movies)
        
        print(f"  Successfully extracted {len(new_movies)} new movies.")
        
        if page < MAX_PAGES:
            print("  Clicking '50 more' to load the next page...")
            try:
                # Need random wait before the action
                random_wait()
                btn = driver.find_element(By.CSS_SELECTOR, "button.ipc-see-more__button")
                driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
                driver.execute_script("arguments[0].click();", btn)
            except Exception as e:
                print("  No '50 more' button found or failed to click. Reached end of results.", e)
                break

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

print(f"\n{'='*70}")
print(f"Total movies collected: {len(all_movies)}")
print("=" * 70)

STEP 3: Dynamic Crawling with Pagination
Navigating to BASE_URL: https://www.imdb.com/search/title/?genres=adventure&explore=genres&title_type=feature&num_votes=25000,&sort=user_rating,desc



Scraping Page 1 of 3
  Waiting 5.5 seconds...


  Scrolling to load images...
  Waiting 2.7 seconds...


  Found 50 movies in the DOM.


  Successfully extracted 50 new movies.
  Clicking '50 more' to load the next page...
  Waiting 8.7 seconds...


...

  Successfully extracted 50 new movies.
  Clicking '50 more' to load the next page...
  Waiting 5.9 seconds...



Scraping Page 3 of 3
  Waiting 9.6 seconds...


  Scrolling to load images...
  Waiting 2.3 seconds...


  Found 150 movies in the DOM.


  Successfully extracted 50 new movies.

Total movies collected: 150


---
## Step 4: Display Results in DataFrame

In [ ]:
if len(all_movies) == 0:
    print("No movie data was scraped. Check output above for errors.")
else:
    df_movies = pd.DataFrame(all_movies)
    df_movies = df_movies[['name', 'runtime', 'certificate', 'rating', 'url']]
    
    print("\n" + "=" * 70)
    print("STEP 4: Display Results")
    print("=" * 70)
    print("\nScraped Movie Data (first 20 rows):")
    display(df_movies.head(20))
    
    print("\n" + "-" * 70)
    print("Summary:")
    print("-" * 70)
    print(f"Total movies scraped: {len(df_movies)}")
    print(f"Pages processed: {MAX_PAGES}")
    print(f"Average movies per page: {len(df_movies) / MAX_PAGES:.1f}")
    
    # Count movies with actual data
    with_names = len(df_movies[df_movies['name'] != 'N/A'])
    with_rating = len(df_movies[df_movies['rating'] != 'N/A'])
    with_runtime = len(df_movies[df_movies['runtime'] != 'N/A'])
    print(f"Movies with names: {with_names}")
    print(f"Movies with ratings: {with_rating}")
    print(f"Movies with runtime: {with_runtime}")


STEP 4: Display Results

Scraped Movie Data (first 20 rows):


,name,runtime,certificate,rating,url
0,1. Attack on Titan: The Last Attack,2h 25m,N/A,9.1,https://www.imdb.com/title/tt33175825/?ref_=sr...
1,2. The Lord of the Rings: The Return of the King,3h 21m,PG-13,9.0,https://www.imdb.com/title/tt0167260/?ref_=sr_t_2
2,3. The Lord of the Rings: The Fellowship of th...,2h 58m,PG-13,8.9,https://www.imdb.com/title/tt0120737/?ref_=sr_t_3
3,4. Inception,2h 28m,PG-13,8.8,https://www.imdb.com/title/tt1375666/?ref_=sr_t_4
4,5. The Lord of the Rings: The Two Towers,2h 59m,PG-13,8.8,https://www.imdb.com/title/tt0167261/?ref_=sr_t_5
5,"6. The Good, the Bad and the Ugly",2h 58m,R,8.8,https://www.imdb.com/title/tt0060196/?ref_=sr_t_6
6,7. Interstellar,2h 49m,PG-13,8.7,https://www.imdb.com/title/tt0816692/?ref_=sr_t_7
7,8. Star Wars: Episode V - The Empire Strikes Back,2h 4m,PG,8.7,https://www.imdb.com/title/tt0080684/?ref_=sr_t_8
8,9. 777 Charlie,2h 44m,N/A,8.7,https://www.imdb.com/title/tt7466810/?ref_=sr_t_9
9,10. Spirited Away,2h 4m,PG,8.6,https://www.imdb.com/title/tt0245429/?ref_=sr_...



----------------------------------------------------------------------
Summary:
----------------------------------------------------------------------
Total movies scraped: 150
Pages processed: 3
Average movies per page: 50.0
Movies with names: 150
Movies with ratings: 150
Movies with runtime: 150


---
## Cleanup

In [ ]:
print("\nClosing browser...")
driver.quit()
print("Done!")


Closing browser...


Done!


---
## Summary

### What Was Done:

1. **Selenium Setup** - Configured Chrome WebDriver with headless mode and anti-detection
2. **Respectful Scraping** - Random 5-10 second waits between actions
3. **Pagination** - Navigated through 3 pages of results (50 movies each)
4. **Data Extraction** - Movie name, runtime, certificate, rating, URL
5. **Display** - Results shown in pandas DataFrame

### Key Techniques:
- Multiple CSS selector fallbacks for robustness
- Headless browser for efficiency
- Scroll handling for lazy-loaded content
- Error handling for missing elements